# Logistic Regression Training

This notebook loads the prepared scaled train and test parquet files from `../data/scaled/`, trains the `logreg` model, saves the fitted model to `../model/`, and writes evaluation outputs to `../model/results/`.


## 1. Imports and Config

Set the shared file locations, output paths, and model-specific dependencies for this training run.


In [1]:
from pathlib import Path

import polars as pl
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
import joblib

DATA_DIR = Path("..") / "data" / "scaled"
X_TRAIN_FILE = DATA_DIR / "X_train.parquet"
Y_TRAIN_FILE = DATA_DIR / "y_train.parquet"
X_TEST_FILE = DATA_DIR / "X_test.parquet"
Y_TEST_FILE = DATA_DIR / "y_test.parquet"
MODEL_DIR = Path("..") / "model"
RESULTS_DIR = MODEL_DIR / "results"
TARGET_COLUMN = "label"
RANDOM_SEED = 42
MODEL_NAME = "logreg"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILE = MODEL_DIR / "logreg_model.joblib"
RESULTS_FILE = RESULTS_DIR / "logreg_results.csv"
METRICS_FILE = RESULTS_DIR / "logreg_metrics.csv"


## 2. Load the Dataset

Load the prepared feature matrices and target vectors from the scaled data folder.


In [2]:
for required_file in [X_TRAIN_FILE, Y_TRAIN_FILE, X_TEST_FILE, Y_TEST_FILE]:
    if not required_file.exists():
        raise FileNotFoundError(f"Required parquet file not found: {required_file}")

X_train = pl.read_parquet(X_TRAIN_FILE)
y_train_df = pl.read_parquet(Y_TRAIN_FILE)
X_test = pl.read_parquet(X_TEST_FILE)
y_test_df = pl.read_parquet(Y_TEST_FILE)

print(f"loaded X_train: {X_train.shape}")
print(f"loaded y_train: {y_train_df.shape}")
print(f"loaded X_test: {X_test.shape}")
print(f"loaded y_test: {y_test_df.shape}")


loaded X_train: (71236, 304)
loaded y_train: (71236, 1)
loaded X_test: (30530, 304)
loaded y_test: (30530, 1)


## 3. Separate Features and Target

Keep the feature matrices separate and extract the binary target from the `label` column in the target parquet files.


In [3]:
y_train = y_train_df.get_column(TARGET_COLUMN).cast(pl.Int32)
y_test = y_test_df.get_column(TARGET_COLUMN).cast(pl.Int32)

print(f"feature columns: {len(X_train.columns)}")
print(f"target column: {TARGET_COLUMN}")


feature columns: 304
target column: label


## 4. Validate the Prepared Features

Fail fast if the upstream split and scaling pipeline did not produce aligned numeric model inputs.


In [4]:
for frame_name, frame in {"y_train": y_train_df, "y_test": y_test_df}.items():
    if TARGET_COLUMN not in frame.columns:
        raise ValueError(f"{frame_name}.parquet must include the target column: {TARGET_COLUMN}")
    if frame.width != 1:
        raise ValueError(f"{frame_name}.parquet should only contain the target column: {TARGET_COLUMN}")

if TARGET_COLUMN in X_train.columns or TARGET_COLUMN in X_test.columns:
    raise ValueError(f"X parquet files should only contain features, not the target column: {TARGET_COLUMN}")

if X_train.columns != X_test.columns:
    raise ValueError("X_train and X_test must contain the same feature columns in the same order.")

if X_train.height != y_train.len():
    raise ValueError("X_train and y_train must have the same number of rows.")

if X_test.height != y_test.len():
    raise ValueError("X_test and y_test must have the same number of rows.")

non_numeric_train = [column_name for column_name, dtype in X_train.schema.items() if not dtype.is_numeric()]
non_numeric_test = [column_name for column_name, dtype in X_test.schema.items() if not dtype.is_numeric()]
non_numeric_features = sorted(set(non_numeric_train + non_numeric_test))

if non_numeric_features:
    preview = ", ".join(non_numeric_features[:10])
    raise ValueError(
        "The split/feature-selection notebook must output model-ready numeric features. "
        f"Non-numeric features found: {preview}"
    )

print(f"validated {len(X_train.columns)} numeric features")


validated 304 numeric features


## 5. Train the Model

Train the logistic regression baseline on the prepared scaled feature matrix.


In [5]:
X_train_np = X_train.to_numpy()
y_train_np = y_train.to_numpy()
X_test_np = X_test.to_numpy()
y_test_np = y_test.to_numpy()

model = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
model.fit(X_train_np, y_train_np)

model


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

## 6. Evaluate and Save

Score the holdout set, save the trained model artifact, and export both metrics and row-level prediction results.


In [6]:
probabilities = model.predict_proba(X_test_np)[:, 1]
predictions = (probabilities >= 0.5).astype(int)
actuals = y_test_np

metrics = {
    "model": MODEL_NAME,
    "accuracy": accuracy_score(actuals, predictions),
    "precision": precision_score(actuals, predictions, zero_division=0),
    "recall": recall_score(actuals, predictions, zero_division=0),
    "roc_auc": roc_auc_score(actuals, probabilities),
}

metrics_frame = pl.DataFrame([metrics])
results_frame = X_test.with_columns(
    [
        pl.Series("actual", actuals),
        pl.Series("predicted", predictions),
        pl.Series("probability", probabilities),
        pl.lit(MODEL_NAME).alias("model"),
    ]
)

joblib.dump(model, MODEL_FILE)
metrics_frame.write_csv(METRICS_FILE)
results_frame.write_csv(RESULTS_FILE)

print(f"model saved to: {MODEL_FILE}")
print(f"metrics saved to: {METRICS_FILE}")
print(f"results saved to: {RESULTS_FILE}")
metrics_frame


model saved to: ..\model\logreg_model.joblib
metrics saved to: ..\model\results\logreg_metrics.csv
results saved to: ..\model\results\logreg_results.csv


model,accuracy,precision,recall,roc_auc
str,f64,f64,f64,f64
"""logreg""",0.637963,0.638542,0.494279,0.691808


## 7. Preview Results

Review a sample of the exported results file before using it for charts or model diagnostics.


In [7]:
results_frame.head(10)


num__age,num__time_in_hospital,num__num_lab_procedures,num__num_procedures,num__num_medications,num__number_outpatient,num__number_emergency,num__number_inpatient,num__number_diagnoses,ord__max_glu_serum,ord__A1Cresult,cat__race_AfricanAmerican,cat__race_Asian,cat__race_Caucasian,cat__race_Hispanic,cat__race_Other,cat__gender_Female,cat__gender_Male,cat__gender_Unknown/Invalid,cat__payer_code_Other,cat__payer_code_Private,cat__payer_code_Public,cat__payer_code_Uninsured,cat__medical_specialty_?,cat__medical_specialty_AllergyandImmunology,cat__medical_specialty_Anesthesiology,cat__medical_specialty_Anesthesiology-Pediatric,cat__medical_specialty_Cardiology,cat__medical_specialty_Cardiology-Pediatric,cat__medical_specialty_DCPTEAM,cat__medical_specialty_Dentistry,cat__medical_specialty_Dermatology,cat__medical_specialty_Emergency/Trauma,cat__medical_specialty_Endocrinology,cat__medical_specialty_Endocrinology-Metabolism,cat__medical_specialty_Family/GeneralPractice,cat__medical_specialty_Gastroenterology,…,cat__admission_source_Sick Baby,cat__admission_source_Transfer from Ambulatory Surgery Center,cat__admission_source_Transfer from a Skilled Nursing Facility (SNF),cat__admission_source_Transfer from a hospital,cat__admission_source_Transfer from another health care facility,cat__admission_source_Transfer from critial access hospital,cat__admission_source_Transfer from hospital inpt/same fac reslt in a sep claim,cat__discharge_disposition_Admitted as an inpatient to this hospital,cat__discharge_disposition_Discharged to home,cat__discharge_disposition_Discharged/transferred to ICF,cat__discharge_disposition_Discharged/transferred to SNF,cat__discharge_disposition_Discharged/transferred to a federal health care facility.,cat__discharge_disposition_Discharged/transferred to a long term care hospital.,cat__discharge_disposition_Discharged/transferred to a nursing facility certified under Medicaid but not certified under Medicare.,cat__discharge_disposition_Discharged/transferred to another rehab fac including rehab units of a hospital .,cat__discharge_disposition_Discharged/transferred to another short term hospital,cat__discharge_disposition_Discharged/transferred to another type of inpatient care institution,cat__discharge_disposition_Discharged/transferred to home under care of Home IV provider,cat__discharge_disposition_Discharged/transferred to home with home health service,cat__discharge_disposition_Discharged/transferred within this institution to Medicare approved swing bed,cat__discharge_disposition_Discharged/transferred/referred another institution for outpatient services,cat__discharge_disposition_Discharged/transferred/referred to a psychiatric hospital of psychiatric distinct part unit of a hospital,cat__discharge_disposition_Discharged/transferred/referred to this institution for outpatient services,cat__discharge_disposition_Expired,cat__discharge_disposition_Expired at home. Medicaid only; hospice.,cat__discharge_disposition_Expired in a medical facility. Medicaid only; hospice.,cat__discharge_disposition_Hospice / home,cat__discharge_disposition_Hospice / medical facility,cat__discharge_disposition_Left AMA,cat__discharge_disposition_NULL,cat__discharge_disposition_Neonate discharged to another hospital for neonatal aftercare,cat__discharge_disposition_Not Mapped,cat__discharge_disposition_Still patient or expected to return for outpatient services,actual,predicted,probability,model
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,i64,f64,str
0.5,-0.5,0.076923,0.5,-0.3,0.0,1.0,0.0,0.0,0.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0